# Paso 5: Análisis de Sensibilidad

---

**Proyecto:** IntApp — Predicción del riesgo de lesión deportiva  
**Autor:** Roberto (Trabajo de Fin de Máster — Fisioterapia Deportiva)  
**Fecha:** Abril 2026

---

## 1. Introducción

El análisis de sensibilidad nos permite responder una pregunta fundamental para tu tesis: **¿qué pasa si cambio los umbrales de riesgo? ¿Es mi sistema robusto o pequeños cambios producen resultados muy diferentes?**

Un sistema clínico es **robusto** cuando sus predicciones no cambian drásticamente ante pequeñas variaciones de los parámetros que tú has fijado. Por ejemplo, si decides que el umbral de riesgo del ratio H:Q es 0.60, pero lo cambias a 0.58 o a 0.62, ¿cambia mucho el rendimiento del modelo? Si la respuesta es no, tu sistema es estable y fiable. Si la respuesta es sí, necesitas justificar muy bien por qué has elegido ese umbral concreto.

En este notebook realizamos **cuatro tipos de análisis de sensibilidad**:

| Análisis | Pregunta que responde |
|---|---|
| **Umbral H:Q ratio** | ¿Cambia el rendimiento si ajusto el punto de corte del ratio isquiotibiales/cuádriceps? |
| **Umbral dolor NRS** | ¿Es crítico el punto de corte del dolor percibido que elegí? |
| **Tamaño muestral N** | ¿Necesito más deportistas o con 500 ya es suficiente? |
| **Importancia de bloques** | ¿Qué bloque de variables (fuerza, movilidad, control, contexto) aporta más al modelo? |

---

> **Para Roberto:** Este análisis es el núcleo del capítulo de **Discusión** de tu TFM. Aquí demuestras que no solo has construido un modelo, sino que has reflexionado críticamente sobre su estabilidad y sus limitaciones. Las preguntas marcadas con **TODO ROBERTO** son las que debes responder con tus propias palabras en la tesis.

## 2. Configuración del entorno e importaciones

In [ ]:
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
warnings.filterwarnings('ignore')

# Añadir la raíz del proyecto al path para poder importar src.*
RAIZ_PROYECTO = Path(os.getcwd()).parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")
print(f"Path actualizado correctamente: {str(RAIZ_PROYECTO) in sys.path}")

In [ ]:
# Importaciones de los módulos del proyecto
from src.generador_datos import generar_dataset
from src.preprocesador import preprocesar
from src.modelo import (
    dividir_datos,
    entrenar_random_forest,
)
from src.variables import (
    COLUMNAS_FUERZA,
    COLUMNAS_MOVILIDAD,
    COLUMNAS_CONTROL,
    COLUMNAS_CONTEXTO,
)
from sklearn.metrics import f1_score

# Directorio de figuras
DIR_FIGURAS = RAIZ_PROYECTO / "figuras"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

print("Importaciones completadas correctamente.")
print(f"Figuras se guardarán en: {DIR_FIGURAS}")

### 2.1 Carga del dataset base y preprocesamiento

Cargamos el dataset sintético ya generado y ejecutamos el pipeline de preprocesamiento completo. Este será nuestro punto de partida para todos los análisis de sensibilidad.

In [ ]:
# Cargar el dataset sintético base (500 deportistas, semilla 42)
RUTA_CSV = RAIZ_PROYECTO / "datos" / "sinteticos" / "dataset_sintetico.csv"
df_crudo = pd.read_csv(RUTA_CSV)

# El generador produce la columna 'riesgo_lesion'; el preprocesador espera 'nivel_riesgo'
if "riesgo_lesion" in df_crudo.columns:
    df_crudo = df_crudo.rename(columns={"riesgo_lesion": "nivel_riesgo"})

print(f"Dataset cargado: {df_crudo.shape[0]} deportistas x {df_crudo.shape[1]} columnas")
print(f"\nDistribución de clases (dataset base):")
print(df_crudo["nivel_riesgo"].value_counts())

In [ ]:
# Preprocesar el dataset base y obtener train/test
df_procesado_base, scaler_base = preprocesar(df_crudo)

X_train_base, X_test_base, y_train_base, y_test_base = dividir_datos(
    df_procesado_base,
    columna_objetivo="nivel_riesgo",
    test_size=0.20,
    semilla=42,
)

# Entrenar el modelo de referencia (Random Forest)
modelo_referencia = entrenar_random_forest(X_train_base, y_train_base, semilla=42)

# F1 macro del modelo de referencia (línea base para comparar)
y_pred_base = modelo_referencia.predict(X_test_base)
f1_referencia = f1_score(
    y_test_base, y_pred_base,
    average="macro",
    labels=["bajo", "medio", "alto"],
    zero_division=0,
)

print(f"\nModelo de referencia entrenado.")
print(f"F1 macro de referencia (umbral NRS=5, umbral HQ default): {f1_referencia:.4f}")
print("\nEste valor será la línea de referencia en todos los gráficos.")

## 3. Sensibilidad del umbral H:Q ratio

### ¿Qué es el ratio H:Q?

El **ratio Hamstring:Quadriceps (H:Q)** es el cociente entre la fuerza de los isquiotibiales (*hamstrings*) y la fuerza del cuádriceps. Es uno de los indicadores biomecánicos más citados en la literatura de prevención de lesiones del ligamento cruzado anterior (LCA).

$$\text{H:Q ratio} = \frac{\text{Fuerza isquiotibiales (N)}}{\text{Fuerza cuádriceps (N)}}$$

**Significado clínico:**
- Un ratio **alto** (isquiotibiales fuertes relativos al cuádriceps) protege la rodilla durante la desaceleración y los cambios de dirección.
- Un ratio **bajo** indica que el cuádriceps domina, lo que aumenta la tensión sobre el LCA.
- El umbral de riesgo más citado en la literatura es **H:Q < 0.60** (Croisier et al., 2008), aunque algunos autores utilizan 0.47 como déficit severo.

### ¿Qué analiza esta sección?

Variamos el umbral de H:Q ratio entre **0.40 y 0.80** en pasos de 0.05. Para cada umbral, regeneramos las etiquetas de riesgo (incluyendo el H:Q como criterio adicional), entrenamos un Random Forest rápido y medimos el F1 macro resultante. Así vemos si el rendimiento del sistema es sensible a la elección de este umbral.

In [ ]:
def generar_label_con_hq(df: pd.DataFrame, umbral_hq: float) -> pd.Series:
    """
    Versión modificada de generar_label que incluye el ratio H:Q como criterio clínico.

    Reglas aplicadas (en orden de prioridad):
      1. Dolor NRS >= 5  → alto (señal directa de alarma clínica)
      2. H:Q ratio < umbral_hq en cualquier lado → medio (riesgo biomecánico)
      3. Historial lesional >= 4 Y exigencia >= 4 → medio (riesgo acumulado)
      4. Valgo dinámico SLS >= 2 en cualquier lado → medio (riesgo biomecánico visual)
      5. Todo lo demás → bajo

    TODO ROBERTO: Ajusta estas reglas según tu criterio clínico y la literatura
    que hayas revisado para tu TFM. El orden de las reglas importa:
    las condiciones más graves deben evaluarse primero.

    Args:
        df: DataFrame crudo con las columnas originales de evaluación.
        umbral_hq: Punto de corte del ratio H:Q. Por debajo → riesgo al menos medio.

    Returns:
        Series con valores 'bajo', 'medio' o 'alto'.
    """
    n = len(df)
    etiquetas = pd.array(["bajo"] * n, dtype=object)

    # Regla 1: Dolor NRS >= 5 → alto
    mascara_alto = df["dolor_percibido_nrs"] >= 5
    etiquetas[mascara_alto.values] = "alto"

    # Regla 2: H:Q ratio por debajo del umbral en cualquier lado → medio
    # El ratio se calcula sobre los datos crudos (isquiotibiales / cuádriceps)
    hq_der = df["isquiotibiales_der"] / df["cuadriceps_der"].replace(0, np.nan)
    hq_izq = df["isquiotibiales_izq"] / df["cuadriceps_izq"].replace(0, np.nan)

    mascara_hq = (
        (hq_der < umbral_hq) | (hq_izq < umbral_hq)
    ) & (~mascara_alto)
    etiquetas[mascara_hq.values] = "medio"

    # Regla 3: Historial alto + exigencia alta → medio (si no está ya clasificado)
    mascara_hist = (
        (df["historial_lesional"] >= 4)
        & (df["perfil_exigencia_deportiva"] >= 4)
        & (~mascara_alto)
        & (~mascara_hq)
    )
    etiquetas[mascara_hist.values] = "medio"

    # Regla 4: Valgo dinámico SLS >= 2 → medio (si no está ya clasificado)
    if "single_leg_squat_valgo_der" in df.columns and "single_leg_squat_valgo_izq" in df.columns:
        mascara_valgo = (
            (df["single_leg_squat_valgo_der"] >= 2)
            | (df["single_leg_squat_valgo_izq"] >= 2)
        ) & (~mascara_alto) & (~mascara_hq) & (~mascara_hist)
        etiquetas[mascara_valgo.values] = "medio"

    return pd.Series(etiquetas, index=df.index, name="nivel_riesgo")

In [ ]:
# Cargar datos crudos (sin renombrar la etiqueta aún, la regeneraremos)
df_crudo_hq = pd.read_csv(RUTA_CSV)

# Rango de umbrales H:Q a evaluar
umbrales_hq = np.arange(0.40, 0.85, 0.05).round(2)

resultados_hq = []

print("Evaluando sensibilidad al umbral H:Q ratio...")
print("-" * 50)

for umbral in umbrales_hq:
    # 1. Regenerar etiquetas con el nuevo umbral H:Q
    df_iter = df_crudo_hq.drop(columns=["riesgo_lesion"], errors="ignore").copy()
    df_iter["nivel_riesgo"] = generar_label_con_hq(df_iter, umbral_hq=umbral)

    # 2. Preprocesar
    df_proc_iter, _ = preprocesar(df_iter)

    # Verificar que hay al menos dos clases (necesario para train_test_split estratificado)
    n_clases = df_proc_iter["nivel_riesgo"].nunique()
    if n_clases < 2:
        print(f"  Umbral {umbral:.2f}: pocas clases ({n_clases}), se omite.")
        continue

    # 3. Dividir y entrenar un RF rápido
    try:
        X_tr, X_te, y_tr, y_te = dividir_datos(
            df_proc_iter, columna_objetivo="nivel_riesgo", test_size=0.20, semilla=42
        )
        modelo_iter = entrenar_random_forest(X_tr, y_tr, semilla=42)
        y_pred_iter = modelo_iter.predict(X_te)
        f1 = f1_score(
            y_te, y_pred_iter,
            average="macro",
            labels=["bajo", "medio", "alto"],
            zero_division=0,
        )
        dist = df_iter["nivel_riesgo"].value_counts().to_dict()
        resultados_hq.append({
            "umbral_hq": umbral,
            "f1_macro": round(f1, 4),
            "n_bajo": dist.get("bajo", 0),
            "n_medio": dist.get("medio", 0),
            "n_alto": dist.get("alto", 0),
        })
        print(f"  Umbral H:Q = {umbral:.2f} → F1 macro = {f1:.4f} "
              f"(bajo={dist.get('bajo',0)}, medio={dist.get('medio',0)}, alto={dist.get('alto',0)})")
    except Exception as e:
        print(f"  Umbral H:Q = {umbral:.2f}: error — {e}")

df_res_hq = pd.DataFrame(resultados_hq)
print("\nAnálisis H:Q completado.")

In [ ]:
# Gráfico: F1 macro vs umbral H:Q ratio
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    df_res_hq["umbral_hq"],
    df_res_hq["f1_macro"],
    marker="o",
    linewidth=2,
    color="#4C72B0",
    markersize=7,
    label="F1 macro (RF)",
)

# Línea de referencia (modelo base sin regla H:Q explícita)
ax.axhline(
    y=f1_referencia,
    color="#DD8452",
    linestyle="--",
    linewidth=1.5,
    label=f"Modelo base (F1={f1_referencia:.4f})",
)

# Marcar el umbral clínico de referencia (0.60, Croisier et al. 2008)
ax.axvline(
    x=0.60,
    color="#55A868",
    linestyle=":",
    linewidth=1.8,
    label="Umbral clínico literatura (0.60)",
)

# Marcar el punto de máximo F1
if not df_res_hq.empty:
    idx_max = df_res_hq["f1_macro"].idxmax()
    umbral_optimo = df_res_hq.loc[idx_max, "umbral_hq"]
    f1_optimo = df_res_hq.loc[idx_max, "f1_macro"]
    ax.annotate(
        f"Máximo: {f1_optimo:.4f}\n(umbral={umbral_optimo:.2f})",
        xy=(umbral_optimo, f1_optimo),
        xytext=(umbral_optimo + 0.05, f1_optimo - 0.03),
        arrowprops=dict(arrowstyle="->", color="#c0392b"),
        fontsize=9,
        color="#c0392b",
    )

ax.set_title(
    "Sensibilidad del F1 macro al umbral H:Q ratio",
    fontsize=13, pad=14,
)
ax.set_xlabel("Umbral H:Q ratio (isquiotibiales / cuádriceps)", fontsize=11)
ax.set_ylabel("F1 macro (Random Forest)", fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_xticks(df_res_hq["umbral_hq"].tolist())
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
RUTA_FIG_HQ = DIR_FIGURAS / "sensibilidad_hq_ratio.png"
fig.savefig(RUTA_FIG_HQ, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada en: {RUTA_FIG_HQ}")

In [ ]:
# Tabla de resultados
print("Tabla de resultados — Sensibilidad al umbral H:Q ratio:")
print("-" * 65)
print(df_res_hq.to_string(index=False))
print("-" * 65)

if not df_res_hq.empty:
    idx_max = df_res_hq["f1_macro"].idxmax()
    umbral_optimo = df_res_hq.loc[idx_max, "umbral_hq"]
    f1_optimo = df_res_hq.loc[idx_max, "f1_macro"]
    variacion = df_res_hq["f1_macro"].max() - df_res_hq["f1_macro"].min()
    print(f"\nUmbral óptimo encontrado: H:Q = {umbral_optimo:.2f} (F1 = {f1_optimo:.4f})")
    print(f"Variación total del F1 en el rango evaluado: {variacion:.4f}")
    if variacion < 0.05:
        print("→ Variación baja (< 0.05): el sistema es ROBUSTO ante cambios en el umbral H:Q.")
    else:
        print("→ Variación alta (>= 0.05): el sistema es SENSIBLE al umbral H:Q elegido.")

### TODO ROBERTO — Preguntas para tu tesis

Después de ver el gráfico y la tabla anteriores, responde estas preguntas en el capítulo de Discusión:

1. **¿En qué valor del umbral H:Q el F1 es máximo?** ¿Coincide ese valor con el umbral de 0.60 que cita la literatura (Croisier et al., 2008)? Si no coincide, ¿qué podría explicar esa discrepancia en tu muestra?

2. **¿La curva es plana o tiene picos pronunciados?** Una curva plana indica robustez (el rendimiento no depende mucho del umbral exacto). Una curva con picos sugiere que la elección del umbral es crítica y debe estar bien justificada bibliográficamente.

3. **¿Qué umbral recomendarías en tu protocolo clínico?** Ten en cuenta que en fisioterapia deportiva suele preferirse ser más conservador (umbral más alto = más deportistas clasificados como riesgo) para no perder casos. Justifica tu decisión.

---

## 4. Sensibilidad del umbral de dolor NRS

### ¿Qué es la escala NRS?

La **Numeric Rating Scale (NRS)** es una escala de 0 a 10 para cuantificar el dolor percibido por el deportista durante actividad con carga:
- **0:** Sin dolor en absoluto
- **5:** Dolor moderado, claramente perceptible
- **10:** El peor dolor imaginable

En el sistema de etiquetado actual, un dolor NRS ≥ 5 clasifica directamente al deportista como **riesgo alto**. Pero, ¿qué pasa si ese umbral lo fijamos en 3, o en 7? Este análisis lo responde.

**Relevancia clínica:** El umbral de dolor es uno de los parámetros más discutidos en la literatura porque depende del tipo de deporte, del momento de la temporada (pretemporada vs. competición) y de la tolerancia individual al dolor del deportista.

In [ ]:
def generar_label_con_nrs(df: pd.DataFrame, umbral_nrs: int) -> pd.Series:
    """
    Versión modificada de generar_label con umbral de dolor NRS parametrizable.

    Reglas aplicadas:
      1. Dolor NRS >= umbral_nrs → alto
      2. Historial lesional >= 4 Y exigencia deportiva >= 4 → medio
      3. Valgo dinámico SLS >= 2 en cualquier lado → medio
      4. Todo lo demás → bajo

    TODO ROBERTO: Ajusta las reglas 2 y 3 según tu criterio clínico y añade
    las reglas adicionales que hayas identificado en la literatura.

    Args:
        df: DataFrame crudo con columnas originales.
        umbral_nrs: Punto de corte NRS por encima del cual → riesgo alto.

    Returns:
        Series con valores 'bajo', 'medio' o 'alto'.
    """
    n = len(df)
    etiquetas = pd.array(["bajo"] * n, dtype=object)

    # Regla 1: Dolor NRS >= umbral_nrs → alto
    mascara_alto = df["dolor_percibido_nrs"] >= umbral_nrs
    etiquetas[mascara_alto.values] = "alto"

    # Regla 2: Historial + exigencia → medio
    mascara_medio = (
        (df["historial_lesional"] >= 4)
        & (df["perfil_exigencia_deportiva"] >= 4)
        & (~mascara_alto)
    )
    etiquetas[mascara_medio.values] = "medio"

    # Regla 3: Valgo dinámico SLS >= 2 → medio
    if "single_leg_squat_valgo_der" in df.columns and "single_leg_squat_valgo_izq" in df.columns:
        mascara_valgo = (
            (df["single_leg_squat_valgo_der"] >= 2)
            | (df["single_leg_squat_valgo_izq"] >= 2)
        ) & (~mascara_alto) & (~mascara_medio)
        etiquetas[mascara_valgo.values] = "medio"

    return pd.Series(etiquetas, index=df.index, name="nivel_riesgo")

In [ ]:
# Rango de umbrales NRS a evaluar (enteros de 3 a 8)
umbrales_nrs = list(range(3, 9))  # [3, 4, 5, 6, 7, 8]

df_crudo_nrs = pd.read_csv(RUTA_CSV)

resultados_nrs = []

print("Evaluando sensibilidad al umbral de dolor NRS...")
print("-" * 55)

for umbral in umbrales_nrs:
    # 1. Regenerar etiquetas con el nuevo umbral NRS
    df_iter = df_crudo_nrs.drop(columns=["riesgo_lesion"], errors="ignore").copy()
    df_iter["nivel_riesgo"] = generar_label_con_nrs(df_iter, umbral_nrs=umbral)

    # 2. Preprocesar
    df_proc_iter, _ = preprocesar(df_iter)

    n_clases = df_proc_iter["nivel_riesgo"].nunique()
    if n_clases < 2:
        print(f"  NRS >= {umbral}: pocas clases ({n_clases}), se omite.")
        continue

    # 3. Dividir y entrenar
    try:
        X_tr, X_te, y_tr, y_te = dividir_datos(
            df_proc_iter, columna_objetivo="nivel_riesgo", test_size=0.20, semilla=42
        )
        modelo_iter = entrenar_random_forest(X_tr, y_tr, semilla=42)
        y_pred_iter = modelo_iter.predict(X_te)
        f1 = f1_score(
            y_te, y_pred_iter,
            average="macro",
            labels=["bajo", "medio", "alto"],
            zero_division=0,
        )
        dist = df_iter["nivel_riesgo"].value_counts().to_dict()
        resultados_nrs.append({
            "umbral_nrs": umbral,
            "f1_macro": round(f1, 4),
            "n_alto": dist.get("alto", 0),
            "n_medio": dist.get("medio", 0),
            "n_bajo": dist.get("bajo", 0),
        })
        print(f"  NRS >= {umbral} → F1 macro = {f1:.4f} "
              f"(alto={dist.get('alto',0)}, medio={dist.get('medio',0)}, bajo={dist.get('bajo',0)})")
    except Exception as e:
        print(f"  NRS >= {umbral}: error — {e}")

df_res_nrs = pd.DataFrame(resultados_nrs)
print("\nAnálisis NRS completado.")

In [ ]:
# Gráfico: F1 macro vs umbral NRS
fig, ax1 = plt.subplots(figsize=(9, 5))

# Eje principal: F1 macro
color_f1 = "#4C72B0"
ax1.plot(
    df_res_nrs["umbral_nrs"],
    df_res_nrs["f1_macro"],
    marker="o",
    linewidth=2,
    color=color_f1,
    markersize=8,
    label="F1 macro (RF)",
)
ax1.axhline(
    y=f1_referencia,
    color="#DD8452",
    linestyle="--",
    linewidth=1.5,
    label=f"Modelo base (F1={f1_referencia:.4f})",
)
ax1.set_xlabel("Umbral NRS (dolor percibido en carga)", fontsize=11)
ax1.set_ylabel("F1 macro (Random Forest)", fontsize=11, color=color_f1)
ax1.set_ylim(0, 1.05)
ax1.set_xticks(umbrales_nrs)
ax1.set_xticklabels([f"NRS ≥ {u}" for u in umbrales_nrs], fontsize=9)
ax1.tick_params(axis="y", labelcolor=color_f1)
ax1.grid(axis="y", linestyle="--", alpha=0.4)

# Eje secundario: número de deportistas clasificados como 'alto'
ax2 = ax1.twinx()
color_n = "#c0392b"
ax2.bar(
    df_res_nrs["umbral_nrs"],
    df_res_nrs["n_alto"],
    alpha=0.20,
    color=color_n,
    label="N deportistas 'alto riesgo'",
    width=0.4,
)
ax2.set_ylabel("N deportistas clasificados como 'alto'", fontsize=10, color=color_n)
ax2.tick_params(axis="y", labelcolor=color_n)

# Leyenda combinada
lineas1, etiq1 = ax1.get_legend_handles_labels()
lineas2, etiq2 = ax2.get_legend_handles_labels()
ax1.legend(lineas1 + lineas2, etiq1 + etiq2, fontsize=9, loc="lower left")

ax1.set_title(
    "Sensibilidad del F1 macro al umbral de dolor NRS",
    fontsize=13, pad=14,
)
ax1.spines["top"].set_visible(False)

fig.tight_layout()
RUTA_FIG_NRS = DIR_FIGURAS / "sensibilidad_nrs.png"
fig.savefig(RUTA_FIG_NRS, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada en: {RUTA_FIG_NRS}")

In [ ]:
# Tabla de resultados
print("Tabla de resultados — Sensibilidad al umbral NRS:")
print("-" * 60)
print(df_res_nrs.to_string(index=False))
print("-" * 60)

if not df_res_nrs.empty:
    variacion_nrs = df_res_nrs["f1_macro"].max() - df_res_nrs["f1_macro"].min()
    umbral_nrs_optimo = df_res_nrs.loc[df_res_nrs["f1_macro"].idxmax(), "umbral_nrs"]
    print(f"\nUmbral NRS óptimo: {umbral_nrs_optimo} (F1 = {df_res_nrs['f1_macro'].max():.4f})")
    print(f"Variación total del F1: {variacion_nrs:.4f}")
    if variacion_nrs < 0.05:
        print("→ Variación baja (< 0.05): el sistema es ROBUSTO ante el umbral NRS.")
    else:
        print("→ Variación alta (>= 0.05): la elección del umbral NRS afecta significativamente al rendimiento.")

### TODO ROBERTO — Preguntas para tu tesis

1. **¿El F1 aumenta o disminuye al elevar el umbral NRS?** Observa también la columna `n_alto` (barras grises): al elevar el umbral, menos deportistas se clasifican como alto riesgo. ¿Qué consecuencia tiene eso en la práctica clínica?

2. **Dilema conservador vs. permisivo:** Un umbral bajo (NRS ≥ 3) captura más deportistas como «alto riesgo» pero puede generar falsas alarmas. Un umbral alto (NRS ≥ 7) es más selectivo pero puede perder casos reales. ¿Cuál es el equilibrio correcto para tu población de estudio?

3. **¿Coincide el umbral óptimo con el que usaste originalmente (NRS ≥ 5)?** Si no coincide, ¿cambiarías tu protocolo?

---

## 5. Sensibilidad al tamaño muestral (curva de aprendizaje)

### ¿Para qué sirve este análisis?

Este análisis responde a la pregunta: **¿es suficiente tener 500 deportistas o necesitaría más datos para que el modelo funcione mejor?**

Una **curva de aprendizaje** muestra cómo evoluciona el rendimiento del modelo a medida que aumentamos el número de muestras de entrenamiento. Si la curva ya se ha estabilizado con 500 deportistas, conseguir más datos no mejoraría mucho el modelo. Si todavía está subiendo, más datos ayudarían significativamente.

Este análisis tiene especial relevancia en tu TFM porque justifica (o cuestiona) el diseño muestral de tu estudio.

In [ ]:
# Tamaños de N a evaluar (de 100 a 1000 en pasos de 100)
tamanios_n = list(range(100, 1100, 100))

resultados_n = []

print("Evaluando sensibilidad al tamaño muestral (curva de aprendizaje)...")
print("-" * 60)

for n in tamanios_n:
    # 1. Generar un dataset nuevo con N deportistas
    df_n = generar_dataset(n_deportistas=n, semilla=42)

    # Renombrar columna de etiqueta
    if "riesgo_lesion" in df_n.columns:
        df_n = df_n.rename(columns={"riesgo_lesion": "nivel_riesgo"})

    # 2. Preprocesar
    df_proc_n, _ = preprocesar(df_n)

    n_clases = df_proc_n["nivel_riesgo"].nunique()
    if n_clases < 2:
        print(f"  N={n}: pocas clases ({n_clases}), se omite.")
        continue

    # 3. Dividir y entrenar
    try:
        X_tr, X_te, y_tr, y_te = dividir_datos(
            df_proc_n, columna_objetivo="nivel_riesgo", test_size=0.20, semilla=42
        )
        modelo_iter = entrenar_random_forest(X_tr, y_tr, semilla=42)
        y_pred_iter = modelo_iter.predict(X_te)
        f1 = f1_score(
            y_te, y_pred_iter,
            average="macro",
            labels=["bajo", "medio", "alto"],
            zero_division=0,
        )
        resultados_n.append({
            "n_deportistas": n,
            "n_entrenamiento": len(X_tr),
            "n_test": len(X_te),
            "f1_macro": round(f1, 4),
        })
        print(f"  N = {n:4d} (train={len(X_tr)}, test={len(X_te)}) → F1 macro = {f1:.4f}")
    except Exception as e:
        print(f"  N={n}: error — {e}")

df_res_n = pd.DataFrame(resultados_n)
print("\nAnálisis de tamaño muestral completado.")

In [ ]:
# Gráfico: F1 macro vs N deportistas (curva de aprendizaje)
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    df_res_n["n_deportistas"],
    df_res_n["f1_macro"],
    marker="s",
    linewidth=2,
    color="#55A868",
    markersize=7,
    label="F1 macro (RF)",
)

# Línea vertical en N=500 (nuestro dataset base)
ax.axvline(
    x=500,
    color="#DD8452",
    linestyle="--",
    linewidth=1.8,
    label="Dataset base (N=500)",
)

# Sombreado de la zona donde el F1 se estabiliza (delta < 0.01 entre pasos consecutivos)
if len(df_res_n) > 1:
    deltas = df_res_n["f1_macro"].diff().abs()
    idx_estable = deltas[deltas < 0.01].index
    if len(idx_estable) > 0:
        n_estable = df_res_n.loc[idx_estable[0], "n_deportistas"]
        ax.axvline(
            x=n_estable,
            color="#4C72B0",
            linestyle=":",
            linewidth=1.5,
            label=f"Inicio de estabilización (~N={n_estable})",
        )

ax.set_title(
    "Curva de aprendizaje — F1 macro vs número de deportistas",
    fontsize=13, pad=14,
)
ax.set_xlabel("Número de deportistas (N)", fontsize=11)
ax.set_ylabel("F1 macro (Random Forest)", fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_xticks(df_res_n["n_deportistas"].tolist())
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
RUTA_FIG_N = DIR_FIGURAS / "sensibilidad_n_deportistas.png"
fig.savefig(RUTA_FIG_N, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada en: {RUTA_FIG_N}")

In [ ]:
# Tabla y análisis de la curva
print("Tabla de resultados — Curva de aprendizaje:")
print("-" * 55)
print(df_res_n.to_string(index=False))
print("-" * 55)

if len(df_res_n) >= 2:
    f1_100 = df_res_n.loc[df_res_n["n_deportistas"] == 100, "f1_macro"].values
    f1_500 = df_res_n.loc[df_res_n["n_deportistas"] == 500, "f1_macro"].values
    f1_1000 = df_res_n.loc[df_res_n["n_deportistas"] == 1000, "f1_macro"].values

    if len(f1_100) > 0 and len(f1_500) > 0:
        ganancia_100_500 = f1_500[0] - f1_100[0]
        print(f"\nGanancia de F1 al pasar de N=100 a N=500: +{ganancia_100_500:.4f}")
    if len(f1_500) > 0 and len(f1_1000) > 0:
        ganancia_500_1000 = f1_1000[0] - f1_500[0]
        print(f"Ganancia de F1 al pasar de N=500 a N=1000: +{ganancia_500_1000:.4f}")
        if abs(ganancia_500_1000) < 0.02:
            print("→ La ganancia es pequeña (< 0.02): 500 deportistas es probablemente suficiente.")
        else:
            print("→ La ganancia es apreciable (>= 0.02): más datos mejorarían el modelo.")

### TODO ROBERTO — Preguntas para tu tesis

1. **¿La curva de aprendizaje se ha estabilizado en N=500 o todavía está subiendo?** Si la curva sigue subiendo claramente, eso significa que tu modelo podría mejorar con más datos. Discute si eso es factible en el contexto real de tu estudio.

2. **¿A partir de qué N el rendimiento deja de mejorar significativamente?** Este es el tamaño muestral mínimo recomendable para tu protocolo. Justifícalo en la sección de Métodos.

3. **Implicación para estudios futuros:** Si el modelo con N=500 sintético ya es bueno, ¿cuántos deportistas reales necesitarías para replicar o superar ese rendimiento? Ten en cuenta que los datos reales son más ruidosos que los sintéticos.

---

## 6. Impacto de eliminar bloques de variables (análisis de ablación)

### ¿Qué es un análisis de ablación?

Un **análisis de ablación** consiste en entrenar el modelo varias veces, eliminando cada vez un grupo de variables, y comparar el rendimiento resultante con el modelo completo. La pérdida de rendimiento al eliminar un bloque nos indica **cuánto aporta ese bloque al modelo**.

Por ejemplo: si al eliminar las variables de fuerza el F1 baja mucho, significa que la fuerza muscular es un predictor muy relevante. Si apenas baja, quizás podrías simplificar tu protocolo de evaluación prescindiendo de esas medidas.

### Los cuatro bloques del sistema IntApp

| Bloque | Variables incluidas | Herramientas de evaluación |
|---|---|---|
| **Fuerza** | Cuádriceps, isquiotibiales, glúteo, aductores, tríceps sural... | Dinamómetro manual |
| **Movilidad** | Dorsiflexión tobillo, extensibilidad isquiotibial, Thomas test... | Goniómetro, WBLT |
| **Control** | Y-Balance Test, single-leg squat valgo, single-leg hop... | Tests funcionales |
| **Contexto** | Edad, género, historial lesional, dolor NRS, estrés/descanso... | Cuestionarios |

In [ ]:
# Identificar qué columnas del DataFrame procesado pertenecen a cada bloque.
# El preprocesamiento añade columnas derivadas (ratios, asimetrías), por lo que
# buscamos por prefijo y raíz en lugar de usar las listas de COLUMNAS_* directamente.

def obtener_columnas_bloque(df_procesado: pd.DataFrame, columnas_originales: list) -> list:
    """
    Dado un DataFrame preprocesado y la lista de columnas originales de un bloque,
    devuelve todas las columnas del DataFrame que están relacionadas con ese bloque:
    - Columnas originales presentes directamente.
    - Columnas derivadas (ratios, asimetrías) cuyo nombre contiene alguna raíz del bloque.

    Args:
        df_procesado: DataFrame tras ejecutar preprocesar().
        columnas_originales: Lista de nombres de columnas del bloque (ej. COLUMNAS_FUERZA).

    Returns:
        Lista de nombres de columnas del DataFrame que pertenecen al bloque.
    """
    # Extraer raíces base (sin sufijo _der/_izq) para detectar columnas derivadas
    raices = set()
    for col in columnas_originales:
        raiz = col.rsplit("_der", 1)[0].rsplit("_izq", 1)[0]
        raices.add(raiz)

    columnas_bloque = []
    for col in df_procesado.columns:
        if col == "nivel_riesgo":
            continue
        # Columna original presente directamente
        if col in columnas_originales:
            columnas_bloque.append(col)
            continue
        # Columna derivada que contiene alguna raíz del bloque
        for raiz in raices:
            if raiz in col:
                columnas_bloque.append(col)
                break

    return list(dict.fromkeys(columnas_bloque))  # eliminar duplicados preservando orden


# Obtener columnas de cada bloque en el DataFrame procesado base
cols_fuerza_proc = obtener_columnas_bloque(df_procesado_base, COLUMNAS_FUERZA)
cols_movilidad_proc = obtener_columnas_bloque(df_procesado_base, COLUMNAS_MOVILIDAD)
cols_control_proc = obtener_columnas_bloque(df_procesado_base, COLUMNAS_CONTROL)
cols_contexto_proc = obtener_columnas_bloque(df_procesado_base, COLUMNAS_CONTEXTO)

print("Columnas por bloque en el DataFrame preprocesado:")
print(f"  Fuerza    : {len(cols_fuerza_proc)} columnas")
print(f"  Movilidad : {len(cols_movilidad_proc)} columnas")
print(f"  Control   : {len(cols_control_proc)} columnas")
print(f"  Contexto  : {len(cols_contexto_proc)} columnas")

In [ ]:
# Definir los cuatro experimentos de ablación
bloques_a_eliminar = {
    "Sin Fuerza": cols_fuerza_proc,
    "Sin Movilidad": cols_movilidad_proc,
    "Sin Control": cols_control_proc,
    "Sin Contexto": cols_contexto_proc,
}

# F1 del modelo completo (referencia)
resultados_ablacion = [
    {
        "experimento": "Modelo completo",
        "bloque_eliminado": "ninguno",
        "n_features": X_train_base.shape[1],
        "f1_macro": round(f1_referencia, 4),
        "delta_f1": 0.0,
    }
]

print("Análisis de ablación por bloques de variables...")
print("-" * 60)
print(f"  Modelo completo ({X_train_base.shape[1]} features) → F1 = {f1_referencia:.4f}")

for nombre_exp, cols_eliminar in bloques_a_eliminar.items():
    # Eliminar las columnas del bloque del dataset preprocesado
    cols_disponibles = [c for c in X_train_base.columns if c not in cols_eliminar]
    cols_disponibles_test = [c for c in X_test_base.columns if c not in cols_eliminar]

    X_tr_abl = X_train_base[cols_disponibles]
    X_te_abl = X_test_base[cols_disponibles_test]

    try:
        modelo_abl = entrenar_random_forest(X_tr_abl, y_train_base, semilla=42)
        y_pred_abl = modelo_abl.predict(X_te_abl)
        f1_abl = f1_score(
            y_test_base, y_pred_abl,
            average="macro",
            labels=["bajo", "medio", "alto"],
            zero_division=0,
        )
        delta = round(f1_abl - f1_referencia, 4)
        resultados_ablacion.append({
            "experimento": nombre_exp,
            "bloque_eliminado": nombre_exp.replace("Sin ", "").lower(),
            "n_features": len(cols_disponibles),
            "f1_macro": round(f1_abl, 4),
            "delta_f1": delta,
        })
        print(f"  {nombre_exp:15s} ({len(cols_disponibles):3d} features) → "
              f"F1 = {f1_abl:.4f}  (Δ = {delta:+.4f})")
    except Exception as e:
        print(f"  {nombre_exp}: error — {e}")

df_res_abl = pd.DataFrame(resultados_ablacion)
print("\nAnálisis de ablación completado.")

In [ ]:
# Gráfico de barras: importancia de cada bloque
fig, ax = plt.subplots(figsize=(9, 5))

# Ordenar por F1 descendente para mejor legibilidad
df_plot = df_res_abl.sort_values("f1_macro", ascending=False)

nombres = df_plot["experimento"].tolist()
f1_vals = df_plot["f1_macro"].tolist()

# Asignar colores: modelo completo en verde, ablaciones en degradado de rojo
paleta = []
for nombre in nombres:
    if nombre == "Modelo completo":
        paleta.append("#55A868")
    elif df_plot.loc[df_plot["experimento"] == nombre, "delta_f1"].values[0] < -0.03:
        paleta.append("#c0392b")  # rojo oscuro: pérdida importante
    else:
        paleta.append("#e67e22")  # naranja: pérdida moderada

barras = ax.bar(nombres, f1_vals, color=paleta, width=0.55, edgecolor="white")

# Etiquetas dentro/encima de las barras
for barra, valor, nombre in zip(barras, f1_vals, nombres):
    delta_row = df_plot.loc[df_plot["experimento"] == nombre, "delta_f1"].values[0]
    delta_str = f"(Δ={delta_row:+.4f})" if nombre != "Modelo completo" else ""
    ax.text(
        barra.get_x() + barra.get_width() / 2.0,
        barra.get_height() + 0.005,
        f"{valor:.4f}\n{delta_str}",
        ha="center", va="bottom",
        fontsize=9, fontweight="bold",
    )

# Línea de referencia del modelo completo
ax.axhline(
    y=f1_referencia,
    color="#55A868",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Modelo completo (F1={f1_referencia:.4f})",
)

ax.set_title(
    "Importancia de cada bloque de variables (análisis de ablación)",
    fontsize=13, pad=14,
)
ax.set_xlabel("Experimento", fontsize=11)
ax.set_ylabel("F1 macro (Random Forest)", fontsize=11)
ax.set_ylim(0, 1.10)
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
RUTA_FIG_ABL = DIR_FIGURAS / "importancia_bloques.png"
fig.savefig(RUTA_FIG_ABL, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada en: {RUTA_FIG_ABL}")

In [ ]:
# Tabla de resultados y ranking de importancia
print("Tabla de resultados — Análisis de ablación por bloques:")
print("-" * 68)
print(df_res_abl.to_string(index=False))
print("-" * 68)

# Ranking por impacto (mayor pérdida = bloque más importante)
df_ranking = (
    df_res_abl[df_res_abl["experimento"] != "Modelo completo"]
    .sort_values("delta_f1", ascending=True)  # más negativo = más importante
    .reset_index(drop=True)
)
df_ranking["importancia_relativa"] = df_ranking["delta_f1"].abs()

print("\nRanking de importancia por bloque (mayor pérdida al eliminar = más importante):")
for i, row in df_ranking.iterrows():
    print(f"  {i+1}. {row['experimento']:15s} → Pérdida de F1: {row['delta_f1']:+.4f}")

### TODO ROBERTO — Preguntas para tu tesis

1. **¿Qué bloque de variables es el más importante para el modelo?** Es el que, al ser eliminado, provoca la mayor caída del F1. ¿Tiene sentido clínico? ¿Coincide con lo que la literatura dice sobre los principales factores de riesgo de lesión?

2. **¿Hay algún bloque cuya eliminación no afecta prácticamente al rendimiento?** Si el F1 casi no cambia al eliminar un bloque (delta cercano a 0), eso sugiere que esas variables son redundantes o poco informativas. ¿Qué implicación tiene eso para el diseño de tu protocolo de evaluación en la práctica clínica?

3. **Implicación clínica para protocolos simplificados:** Si tienes que evaluar a muchos deportistas con tiempo limitado, ¿qué bloque omitirías primero según estos resultados? ¿Es esa una decisión clínicamente razonable?

---

## 7. Resumen del análisis de sensibilidad

### Tabla resumen de todos los análisis realizados

| Análisis | Parámetro variado | Rango evaluado | Variación F1 observada | Conclusión preliminar |
|---|---|---|---|---|
| H:Q ratio | Umbral isquiotibiales/cuádriceps | 0.40 – 0.80 | *Ver celda anterior* | *Completar tras ejecutar* |
| Dolor NRS | Umbral de dolor en carga | NRS ≥ 3 a NRS ≥ 8 | *Ver celda anterior* | *Completar tras ejecutar* |
| Tamaño muestral | N deportistas | 100 – 1000 | *Ver celda anterior* | *Completar tras ejecutar* |
| Ablación de bloques | Bloque eliminado | Fuerza / Movilidad / Control / Contexto | *Ver celda anterior* | *Completar tras ejecutar* |

> **Instrucción para Roberto:** Rellena la columna «Conclusión preliminar» con los valores reales que hayas obtenido tras ejecutar el notebook. Luego usa esa tabla en el capítulo de Discusión de tu TFM.

In [ ]:
# Tabla resumen automática con los valores reales obtenidos
print("=" * 70)
print("  RESUMEN DEL ANÁLISIS DE SENSIBILIDAD")
print("=" * 70)

print(f"\nModelo de referencia (Random Forest, N=500):")
print(f"  F1 macro base: {f1_referencia:.4f}")

print("\n--- 1. Umbral H:Q ratio ---")
if not df_res_hq.empty:
    var_hq = df_res_hq["f1_macro"].max() - df_res_hq["f1_macro"].min()
    opt_hq = df_res_hq.loc[df_res_hq["f1_macro"].idxmax(), "umbral_hq"]
    print(f"  Rango F1: [{df_res_hq['f1_macro'].min():.4f}, {df_res_hq['f1_macro'].max():.4f}]")
    print(f"  Variación total: {var_hq:.4f}")
    print(f"  Umbral óptimo: H:Q = {opt_hq:.2f}")
    print(f"  Robustez: {'ALTA (variación < 0.05)' if var_hq < 0.05 else 'BAJA (variación >= 0.05)'}")

print("\n--- 2. Umbral dolor NRS ---")
if not df_res_nrs.empty:
    var_nrs = df_res_nrs["f1_macro"].max() - df_res_nrs["f1_macro"].min()
    opt_nrs = df_res_nrs.loc[df_res_nrs["f1_macro"].idxmax(), "umbral_nrs"]
    print(f"  Rango F1: [{df_res_nrs['f1_macro'].min():.4f}, {df_res_nrs['f1_macro'].max():.4f}]")
    print(f"  Variación total: {var_nrs:.4f}")
    print(f"  Umbral óptimo: NRS >= {opt_nrs}")
    print(f"  Robustez: {'ALTA (variación < 0.05)' if var_nrs < 0.05 else 'BAJA (variación >= 0.05)'}")

print("\n--- 3. Tamaño muestral ---")
if not df_res_n.empty:
    var_n = df_res_n["f1_macro"].max() - df_res_n["f1_macro"].min()
    f1_n500 = df_res_n.loc[df_res_n["n_deportistas"] == 500, "f1_macro"]
    f1_n1000 = df_res_n.loc[df_res_n["n_deportistas"] == 1000, "f1_macro"]
    print(f"  Rango F1: [{df_res_n['f1_macro'].min():.4f}, {df_res_n['f1_macro'].max():.4f}]")
    if len(f1_n500) > 0 and len(f1_n1000) > 0:
        ganancia = f1_n1000.values[0] - f1_n500.values[0]
        print(f"  Ganancia al doblar N (500→1000): {ganancia:+.4f}")
        print(f"  Suficiencia de N=500: {'SI (ganancia < 0.02)' if abs(ganancia) < 0.02 else 'NO (ganancia >= 0.02, más datos ayudarían)'}")

print("\n--- 4. Ablación de bloques ---")
if not df_ranking.empty:
    print("  Ranking de importancia (de mayor a menor):")
    for i, row in df_ranking.iterrows():
        print(f"  {i+1}. {row['experimento']:15s} Δ F1 = {row['delta_f1']:+.4f}")
    bloque_mas_importante = df_ranking.iloc[0]["experimento"]
    print(f"\n  Bloque más importante: {bloque_mas_importante}")

print("\n" + "=" * 70)

## TODO ROBERTO — Preguntas finales para el capítulo de Discusión

---

Estas son las preguntas centrales que debes responder en tu TFM a partir de todos los análisis de este notebook. No basta con copiar los números: debes interpretarlos clínicamente.

---

### Pregunta 1: ¿Tu sistema de scoring es robusto ante variaciones de umbrales?

Un sistema es robusto si pequeñas variaciones en los parámetros (umbral H:Q, umbral NRS) producen variaciones pequeñas en el rendimiento (F1). Comenta el rango de variación del F1 que has observado en las secciones 3 y 4, y concluye si tu sistema es estable o si, por el contrario, la elección de los umbrales es crítica y debe justificarse cuidadosamente.

*Orientación:* Si la variación del F1 al cambiar el umbral H:Q entre 0.40 y 0.80 es menor de 0.05 puntos, el sistema es robusto. Si supera 0.10, la elección del umbral es determinante y necesitas apoyarla fuertemente en la literatura.

---

### Pregunta 2: ¿Qué conclusiones sacas para la práctica clínica?

El análisis de ablación (sección 6) te dice qué bloque de variables es más informativo. Usa ese resultado para recomendar qué evaluaciones son imprescindibles en un protocolo de cribado y cuáles podrían ser opcionales si el tiempo o los recursos son limitados.

Por ejemplo: *«Los resultados sugieren que el bloque de control motor aporta la mayor parte de la capacidad predictiva del modelo. En protocolos de cribado rápido, este bloque debería priorizarse frente al bloque de movilidad, cuya eliminación produjo una pérdida menor de rendimiento.»*

---

### Pregunta 3: ¿Recomendarías umbrales más conservadores o más agresivos?

En fisioterapia deportiva, el coste de un **falso negativo** (no detectar un deportista en riesgo real) suele ser mayor que el de un **falso positivo** (sobrediagnosticar). Esto habla a favor de umbrales conservadores (más bajos para el dolor, H:Q más alto para marcar riesgo). Sin embargo, umbrales demasiado conservadores generan un número excesivo de deportistas clasificados como «alto riesgo», lo que puede desbordar la capacidad asistencial.

Formula tu recomendación concreta y justifícala:
- ¿Qué umbral de H:Q ratio recomendarías en tu protocolo?
- ¿Qué umbral de NRS recomendarías?
- ¿Cómo influye en esa decisión el contexto del deporte (pretemporada vs. mitad de temporada)?

---

### Pregunta 4: ¿Cuántos deportistas reales necesitarías para validar el sistema?

A partir de la curva de aprendizaje (sección 5), argumenta cuántas muestras reales serían necesarias para replicar el rendimiento observado con datos sintéticos. Recuerda que los datos reales son más ruidosos (más variabilidad inter-evaluador, datos ausentes, etc.), por lo que probablemente necesites más deportistas reales que el equivalente sintético.

---

> **Nota metodológica:** Todos los análisis de este notebook se basan en datos **sintéticos**. Los resultados deben interpretarse como evidencia del comportamiento *teórico* del sistema. La validez clínica real solo puede establecerse con datos reales de pacientes, validación externa y, idealmente, un ensayo prospectivo. Esta limitación debe mencionarse explícitamente en la sección de Limitaciones de tu TFM.

---